# Análise de sentimentos em português

Objetivo: reunir em um único notebook a exploração dos dados, os resultados dos modelos e a análise de erros.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from sentiment_analysis.data import load_splits, prepare_dataset

try:
    splits = load_splits('quick')
except FileNotFoundError:
    prepare_dataset('quick')
    splits = load_splits('quick')

In [ ]:
data = pd.concat(splits.values(), keys=splits, names=['split']).reset_index(level=0)
display(data.groupby(['split', 'label']).size().unstack(fill_value=0))
display(data.assign(characters=data.text.str.len()).groupby('label').characters.describe().round(1))

## Metodologia

As notas 1–2 viram `negative`, 3 vira `neutral` e 4–5 viram `positive`. O modo quick usa 1.200 textos por classe e split estratificado 64/16/20. TF-IDF é ajustado somente no treino; hiperparâmetros são escolhidos na validação e o teste permanece isolado.

In [ ]:
comparison = pd.read_csv(ROOT / 'reports' / 'model_comparison.csv')
comparison

## Conclusões

A Regressão Logística foi o melhor modelo no teste quick (Macro F1 0,7504). O Transformer zero-shot teve dificuldade com a classe neutra, evidenciando mudança de domínio. Casos mistos, negação, ironia, textos curtos e o uso de estrelas como rótulo explicam parte dos erros. Métricas GenAI permanecem N/A porque nenhum provedor real foi executado.